In [1]:
%pwd

'/Users/pardhu/Developer/TMF Classfier/research'

In [2]:
cd ..

/Users/pardhu/Developer/TMF Classfier


In [3]:
%pwd

'/Users/pardhu/Developer/TMF Classfier'

In [6]:
from pathlib import Path
import pandas as pd
import fitz  # PyMuPDF
from docx import Document
from tqdm import tqdm

In [7]:
DATA_DIR = Path("data")

classes = [
    "protocol",
    "informed_consent",
    "statistical_analysis_plan",
    "safety_report"
]

In [8]:
def extract_pdf_text(pdf_path):
    
    text = ""

    try:
        doc = fitz.open(pdf_path)

        for page in doc:
            text += page.get_text()

        doc.close()

    except Exception as e:
        print(f"Error reading {pdf_path}: {e}")

    return text

In [9]:
def extract_docx_text(docx_path):

    text = ""

    try:
        doc = Document(docx_path)

        for para in doc.paragraphs:
            text += para.text + "\n"

    except Exception as e:
        print(f"Error reading {docx_path}: {e}")

    return text

In [10]:
def extract_text(file_path):

    suffix = file_path.suffix.lower()

    if suffix == ".pdf":
        return extract_pdf_text(file_path)

    elif suffix in [".docx", ".doc"]:
        return extract_docx_text(file_path)

    elif suffix == ".txt":

        try:
            return file_path.read_text(
                encoding="utf-8",
                errors="ignore"
            )

        except Exception as e:
            print(e)
            return ""

    return ""

In [14]:
dataset = []

for cls in classes:

    folder = DATA_DIR / cls

    files = list(folder.glob("*"))

    files = [
        f for f in files
        if not f.name.startswith(".")
    ]

    print(f"\nProcessing {cls}...")

    for file in tqdm(files):

        text = extract_text(file)

        # Skip files with failed extraction
        if len(text.strip()) < 100:
            print(f"Skipping {file.name} (empty/failed extraction)")
            continue

        dataset.append({
            "file_name": file.name,
            "class": cls,
            "text": text,
            "num_chars": len(text)
        })


Processing protocol...


100%|██████████| 15/15 [00:15<00:00,  1.03s/it]



Processing informed_consent...


 38%|███▊      | 5/13 [00:00<00:00, 29.23it/s]

Error reading data/informed_consent/IRB consent template - social-behavioral.docx: file 'data/informed_consent/IRB consent template - social-behavioral.docx' is not a Word file, content type is 'application/vnd.openxmlformats-officedocument.themeManager+xml'
Skipping IRB consent template - social-behavioral.docx (empty/failed extraction)
Error reading data/informed_consent/ethics-informedconsent-clinicalstudies.docx: file 'data/informed_consent/ethics-informedconsent-clinicalstudies.docx' is not a Word file, content type is 'application/vnd.openxmlformats-officedocument.themeManager+xml'
Skipping ethics-informedconsent-clinicalstudies.docx (empty/failed extraction)
Error reading data/informed_consent/IRB consent template - biomedical.docx: file 'data/informed_consent/IRB consent template - biomedical.docx' is not a Word file, content type is 'application/vnd.openxmlformats-officedocument.themeManager+xml'
Skipping IRB consent template - biomedical.docx (empty/failed extraction)


...

 73%|███████▎  | 11/15 [00:05<00:01,  2.13it/s]

MuPDF error: format error: No default Layer config



 87%|████████▋ | 13/15 [00:06<00:00,  2.30it/s]

MuPDF error: format error: No default Layer config



100%|██████████| 15/15 [00:06<00:00,  2.18it/s]


In [15]:
df = pd.DataFrame(dataset)

print("Total Valid Documents:", len(df))

df.head()

Total Valid Documents: 53


,file_name,class,text,num_chars
0,Prot_008.pdf,protocol,Clinical Study Protocol \nDrug Substance Durva...,271411
1,Prot_009.pdf,protocol,Official Protocol Title: \nNCT number: \nNCT03...,358803
2,Prot_007.pdf,protocol,Official Protocol Title:\nNCT number:\nDocumen...,187601
3,Prot_013.pdf,protocol,PF-06863135 (Elranatamab)\nProtocol C1071003\n...,342031
4,Prot_012.pdf,protocol,"This may include, but is not limited to, redac...",212928


In [16]:
df["class"].value_counts()

class
protocol                     15
safety_report                15
statistical_analysis_plan    14
informed_consent              9
Name: count, dtype: int64

In [17]:
df.groupby("class").agg(
    total_docs=("file_name", "count"),
    avg_chars=("num_chars", "mean"),
    min_chars=("num_chars", "min"),
    max_chars=("num_chars", "max")
)

,total_docs,avg_chars,min_chars,max_chars
class,,,,
informed_consent,9,12412.333333,1576,44805
protocol,15,225965.666667,81088,358803
safety_report,15,94646.066667,3231,647538
statistical_analysis_plan,14,155246.857143,8844,305764


In [18]:
df.to_csv(
    "raw_extracted_dataset.csv",
    index=False
)

print("Clean dataset saved successfully.")

Clean dataset saved successfully.
